# `scVI` correction, leiden clustering and manual annotation of the neuronal cells

**Developed by:** Anna Maguza\
**Affiliation:** Faculty of Medicine, Würzburg University\
**Creation date:** 3rd February 2025\
**Last modified date:** 3rd February 2025

#### **Objective**

This notebook continues the process for annotation of mesenchymal cell states. We already annotated cells by label transfer and validated predicted cell states by labels transfer, the purpose of the next steps are:
1. Annnotate subpopulations of Branch A and Branch B cells
2. Annnotate subpopulations of progenitor and glial cells
3. Annnotate subpopulations of neuroblast cells

## Import packages

In [ ]:
import scvi
import anndata
import warnings
import numpy as np
import scanpy as sc
import anndata
import pandas as pd
import plotnine as p
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
from scib_metrics.benchmark import Benchmarker

import json
from datetime import datetime

## Setup working environment

In [2]:
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi = 180, color_map = 'magma_r', dpi_save = 300, vector_friendly = True, format = 'svg')

In [ ]:
warnings.simplefilter(action = 'ignore')
scvi.settings.seed = 1712
%config InlineBackend.print_figure_kwargs = {'facecolor' : "w"}
%config InlineBackend.figure_format = 'retina'

In [4]:
arches_params = dict(
    use_layer_norm = "both",
    use_batch_norm = "none",
    encode_covariates = True,
    dropout_rate = 0.2,
    n_layers = 3,
)

In [5]:
def X_is_raw(adata):
    return np.array_equal(adata.X.sum(axis=0).astype(int), adata.X.sum(axis=0))

In [6]:
timestamp = datetime.now().strftime('%d%m%Y_%H%M%S')

In [ ]:
timestamp

## Upload data

In [ ]:
adata = sc.read_h5ad('data/gut_data/gut_hs_all_datasets_scVI_scANVI_neuronal_cellstates_AM_03022025_184954_raw.h5ad')
adata

In [9]:
adata.obs.index = adata.obs.index.astype(str)

In [10]:
adata.obs_names_make_unique()

In [ ]:
duplicated_obs = adata.obs.index.duplicated().sum()
print(f"Number of duplicated observation indexes: {duplicated_obs}")

In [12]:
region_map = {
    'duodenum': 'small intestine',
    'ileum': 'small intestine',
    'small intestine': 'small intestine',
    'terminal ileum': 'small intestine',
    'jejunum': 'small intestine',
    'sigmoid colon': 'large intestine',
    'caecum': 'large intestine',
    'ascending colon': 'large intestine',
    'transverse colon': 'large intestine',
    'large intestine': 'large intestine',
    'descending colon': 'large intestine',
    'appendix': 'large intestine',
    'colon': 'large intestine',
    'rectum': 'large intestine'
}

adata.obs['gut_region'] = adata.obs['organism_part'].map(region_map)

In [13]:
adata.obs['cell_states'] = adata.obs['cellstates_scANVI'].copy()

## Progenitors and glial

In [ ]:
adata.obs['cellstates_scANVI'].value_counts()

In [ ]:
adata_glial = adata[adata.obs['cellstates_scANVI'].isin(['Progenitor', 'Neuroblast', 'Glia'])]
adata_glial

### Extract highly variable genes

In [31]:
adata_glial.layers['counts'] = adata_glial.X.copy()

In [ ]:
sc.pp.highly_variable_genes(
    adata_glial,
    flavor = "seurat_v3",
    n_top_genes = 1000,
    layer = "counts",
    batch_key = "library_preparation_protocol",
    subset = True,
    span = 1
)

## Run scVI

In [33]:
scvi.model.SCVI.setup_anndata(adata_glial, 
                              categorical_covariate_keys=['sample_id', 'library_construnction_and_layout', 'Protocol REF'],
                              layer = 'counts')

In [34]:
scvi_model = scvi.model.SCVI(adata_glial,
                            n_latent = 50, 
                            n_hidden = 128,
                            n_layers = 2, 
                            dropout_rate = 0.1,
                            dispersion = 'gene-batch', 
                            gene_likelihood = 'nb')

In [ ]:
scvi_model.train(100, 
                early_stopping = True,
                early_stopping_patience = 10,
                check_val_every_n_epoch = 1, 
                enable_progress_bar = True)

In [36]:
adata_glial.obsm["X_scVI"] = scvi_model.get_latent_representation(adata_glial)

#### Evaluate model performance using the [_Svensson_](https://www.nxn.se/valent/2023/8/10/training-scvi-posterior-predictive-distributions-over-epochs) method

In [37]:
history_df = (
    scvi_model.history['elbo_train'].astype(float)
    .join(scvi_model.history['elbo_validation'].astype(float))
    .reset_index()
    .melt(id_vars=['epoch'])
)

In [ ]:
plt.figure(figsize=(12, 10))

plt.plot(history_df[history_df['variable'] == 'elbo_train']['epoch'], 
         history_df[history_df['variable'] == 'elbo_train']['value'], 
         color='black', marker='o', label='Training ELBO')

plt.plot(history_df[history_df['variable'] == 'elbo_validation']['epoch'],
         history_df[history_df['variable'] == 'elbo_validation']['value'], 
         color='red', marker='o', label='Validation ELBO')

plt.xlabel('Epoch')
plt.ylabel('ELBO Value')
plt.title('Training and Validation ELBO over Epochs')
plt.legend()
plt.grid(True, alpha=0.3)

plt.xlim(left=1)

plt.tight_layout()
plt.show()

+ Visualize dataset

In [ ]:
sc.pp.neighbors(adata_glial, use_rep = "X_scVI", n_neighbors = 50, metric = 'minkowski')
sc.tl.umap(adata_glial, min_dist = 0.3, spread = 4, random_state = 1712)

### `Leiden` clustering

In [ ]:
sc.tl.leiden(adata_glial, resolution = 0.4, random_state = 1786)

In [ ]:
sc.pl.umap(adata_glial,color=['leiden', 'Integrated_05', 'cellstates_scANVI'], color_map = 'magma_r', ncols=2, frameon=False, show=False, size = 3)

## Manual Annotation

In [ ]:
adata_log = adata[adata.obs['cellstates_scANVI'].isin(['Progenitor', 'Neuroblast', 'Glia'])].copy()
adata_log.obs['leiden'] = adata_glial.obs['leiden'].copy()
adata_log.obsm['X_umap'] = adata_glial.obsm['X_umap'].copy()
sc.pp.normalize_total(adata_log, target_sum=1e6, exclude_highly_expressed=True)
sc.pp.log1p(adata_log)

* Enteric glia

In [ ]:
sc.pl.umap(adata_log,color=['leiden', 'S100B', 'CRYAB', 'MPZ', 'cell_states'], ncols=4, frameon=False, show=False, size = 3)

* Glial 1

In [ ]:
sc.pl.umap(adata_log,color=['leiden', 'DHH', 'RXRG', 'NTRK2', 'MBP', 'cell_states'], ncols=4, frameon=False, show=False, size = 3)

* Glial 2

In [ ]:
sc.pl.umap(adata_log,color=['leiden', 'ELN', 'TFAP2A', 'SOX8', 'BMP8B', 'cell_states'], ncols=4, frameon=False, show=False, size = 3)

* Glial 3

In [ ]:
sc.pl.umap(adata_log,color=['leiden', 'BCAN', 'APOE', 'CALCA', 'FRZB', 'cell_states'], ncols=4, frameon=False, show=False, size = 3)

* Differentiating glia

In [ ]:
sc.pl.umap(adata_log,color=['leiden', 'COL20A1', 'cell_states'], ncols=4, frameon=False, show=False, size = 3)

* Terminal glial cells - FOXD3, MPZ, CDH19, PLP1, SOX10, S100B and ERBB3, but lacked RET

In [ ]:
sc.pl.umap(adata_log,color=['leiden', 'FOXD3', 'MPZ', 'CDH19', 'PLP1', 'SOX10', 'S100B', 'ERBB3', 'RET', 'cell_states'], ncols=4, frameon=False, show=False, size = 3)

* Glial progenitors

In [ ]:
sc.pl.umap(adata_log,color=['leiden', 'PHOX2B', 'age_group', 'cell_states'], ncols=4, frameon=False, show=False, size = 3)

* Intra-ganglionic glia

In [ ]:
sc.pl.umap(adata_log,color=['leiden', 'ENTPD2', 'age_group', 'cell_states'], ncols=4, frameon=False, show=False, size = 3)

* Submucosal glia

In [ ]:
sc.pl.umap(adata_log,color=['leiden', 'TGFB1', 'age_group', 'cell_states'], ncols=4, frameon=False, show=False, size = 3)

* Submucosal precursor

In [ ]:
sc.pl.umap(adata_log,color=['leiden', 'HAND2', 'age_group', 'cell_states'], ncols=4, frameon=False, show=False, size = 3)

* Lymphoid associated precursor

In [ ]:
sc.pl.umap(adata_log,color=['leiden', 'FGL2', 'MAL', 'TGFBR3', 'GFRA3', 'ARTN', 'RXRG', 'age_group', 'cell_states'], ncols=4, frameon=False, show=False, size = 3)

* Neuroblast

In [ ]:
sc.pl.umap(adata_log,color=['leiden', 'ASCL1', 'SOX11', 'NEUROD1', 'DCX', 'STMN2', 'GAP43', 'age_group', 'cell_states'], ncols=4, frameon=False, show=False, size = 3)

* Glial Progenitors

In [ ]:
sc.pl.umap(adata_log,color=['leiden', 'SOX10', 'PLP1', 'S100B', 'FABP7', 'ERBB3', 'age_group', 'cell_states'], ncols=4, frameon=False, show=False, size = 3)

* ENS Progenitors

In [ ]:
sc.pl.umap(adata_log,color=['leiden', 'SOX10', 'PHOX2B', 'RET', 'ASCL1', 'EDNRB', 'age_group', 'cell_states'], ncols=4, frameon=False, show=False, size = 3)

* Neuronal branch A

In [ ]:
sc.pl.umap(adata_log,color=['leiden','ETV1', 'cell_states'], ncols=4, frameon=False, show=False, size = 3)

* Branch A1 - inhibitory motor neurons (iMN)

In [ ]:
sc.pl.umap(adata_log,color=['leiden', 'GAL', 'NOS1', 'VIP', 'cell_states'], ncols=4, frameon=False, show=False, size = 3)

* Branch A2 - mixed IPAN/IN population based 

In [ ]:
sc.pl.umap(adata_log,color=['leiden', 'NTNG1', 'NXPH2', 'cell_states'], ncols=4, frameon=False, show=False, size = 3)

* Branch A4 -  interneurons (IN)

In [ ]:
sc.pl.umap(adata_log,color=['leiden', 'NEUROD6', 'cell_states'], ncols=4, frameon=False, show=False, size = 3)

* Neuronal branch B

In [ ]:
sc.pl.umap(adata_log,color=['leiden', 'BNC2', 'cell_states'], ncols=4, frameon=False, show=False, size = 3)

* B1 -  immature excitatory motor neurons (eMN)

In [ ]:
sc.pl.umap(adata_log,color=['leiden', 'NXPH4', 'NDUFA4L2','cell_states'], ncols=4, frameon=False, show=False, size = 3)

* B2 - second cluster of eMN

In [ ]:
sc.pl.umap(adata_log,color=['leiden', 'PENK', 'cell_states'], ncols=4, frameon=False, show=False, size = 3)

* B3 (IPAN)

In [ ]:
sc.pl.umap(adata_log,color=['leiden', 'DLX3', 'ANO2', 'NOG', 'NTRK3','SST', 'cell_states'], ncols=4, frameon=False, show=False, size = 3)

In [94]:
#adata_log.obs.loc[adata_log.obs['leiden'] == '0', 'cell_states'] = 'Stromal 1 (ADAMDEC1+)'
#adata_log.obs.loc[adata_log.obs['leiden'] == '7', 'cell_states'] = 'Stromal 1 (ADAMDEC1+)'

In [113]:
all_categories = pd.Categorical(
    pd.concat([adata.obs['cell_states'], adata_log.obs['cell_states']]).unique()
)

adata.obs['cell_states'] = pd.Categorical(
    adata.obs['cell_states'],
    categories=all_categories
)

adata_log.obs['cell_states'] = pd.Categorical(
    adata_log.obs['cell_states'],
    categories=all_categories
)

In [114]:
shared_indices = adata_log.obs.index
adata.obs.loc[shared_indices, 'cell_states'] = adata_log.obs['cell_states']

In [116]:
#del adata_stromal, adata_log

## Save prepared dataset

In [ ]:
current_history = adata.uns['processing_history'].tolist()

new_entry = json.dumps({
    'timestamp': timestamp,
    'step': 'Manually annotated cell states',
})
current_history.append(new_entry)

adata.uns['processing_history'] = current_history

In [228]:
adata.obs.rename(columns={'cell_id': 'cell_index'}, inplace=True)

In [229]:
project = 'gut'
species = 'hs'
name = 'AM'
counts = 'raw'
atribute = 'all_datasets_mesenchymal_annotated'

adata.write_h5ad(f"data/gut_data/{project}_{species}_{atribute}_{name}_{timestamp}_{counts}.h5ad")